# 01. Build variant-level feature matrix

This notebook starts from the already prepared `master_snv_table_v1.csv` and creates several variant-level matrices for clustering.

Main idea:

- one row = one mtDNA variant
- `variant_id` = variant identifier
- modeling columns = numeric features only
- labels such as `validation_label`, `is_pathogenic_dataset9`, `is_neutral_dataset8`, and `is_artifact_prone_site` are not used in the primary unsupervised matrix

## 1. Imports and paths

In [58]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

INPUT_PATH = Path("../data/processed/master_snv_table_v1.tsv")
OUTPUT_DIR = Path("../data/processed/neutral_scoring_prepared")
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_TABLE = OUTPUT_DIR / "neutral_scoring_input_v1.tsv"
FEATURE_SETS_REPORT = OUTPUT_DIR / "neutral_scoring_feature_sets_v1.txt"
LABEL_AUDIT = OUTPUT_DIR / "neutral_scoring_label_audit_v1.tsv"

## 2. Load master file

In [59]:
master_table = pd.read_csv(INPUT_PATH, sep='\t', low_memory=False)
master_table.head()

,position,reference,alternate,consequence,mlc_score,variant_id,mlc_position_score,gnomad_observed,gnomad_filter,AN,gnomad_homoplasmic_ac,gnomad_heteroplasmic_ac,gnomad_homoplasmic_af,gnomad_heteroplasmic_af,gnomad_combined_af_simple,gnomad_max_heteroplasmy,hap_defining_variant,vep,feature,gene,helix_counts_hom,helix_af_hom,helix_counts_het,helix_af_het,helix_mean_arf,helix_max_arf,helix_haplogroups_hom,helix_haplogroups_het,helix_observed,is_disease_suspected_dataset3,is_neutral_dataset8,is_pathogenic_dataset9,validation_label,gene_constraint_symbol,gene_constraint_start_position,gene_constraint_end_position,gene_constraint_consequence,gene_constraint_observed,gene_constraint_expected,gene_constraint_obs_exp,gene_constraint_lower_ci,gene_constraint_upper_ci,regional_constraint_symbol,regional_constraint_start_position,regional_constraint_end_position,regional_constraint_protein_residue_start,regional_constraint_protein_residue_end,regional_constraint_observed,regional_constraint_expected,regional_constraint_obs_exp,regional_constraint_lower_ci,regional_constraint_upper_ci,in_regional_constraint,noncoding_constraint_locus,noncoding_constraint_description,noncoding_constraint_start_position,noncoding_constraint_end_position,noncoding_constraint_observed,noncoding_constraint_expected,noncoding_constraint_obs_exp,noncoding_constraint_lower_ci,noncoding_constraint_upper_ci,in_noncoding_constraint,is_artifact_prone_site
0,1,G,T,intergenic_variant,0.65755,m.1G>T,0.65755,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,unlabeled,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,1,G,A,intergenic_variant,0.65755,m.1G>A,0.65755,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,unlabeled,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,1,G,C,intergenic_variant,0.65755,m.1G>C,0.65755,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,unlabeled,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,2,A,T,intergenic_variant,0.64832,m.2A>T,0.64832,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,unlabeled,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,2,A,C,intergenic_variant,0.64832,m.2A>C,0.64832,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,unlabeled,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0


In [60]:
print("Shape:", master_table.shape)

Shape: (49704, 64)


In [61]:
id_cols = ["variant_id", "position", "reference", "alternate"]

label_cols = ["validation_label", "is_disease_suspected_dataset3", "is_neutral_dataset8", "is_pathogenic_dataset9"]

required_basic_cols = id_cols + label_cols

missing_basic_cols = [col for col in required_basic_cols if col not in master_table.columns]

if missing_basic_cols:

    raise ValueError(f"Missing expected basic columns: {missing_basic_cols}")

print("\nValidation label distribution:")

print(master_table["validation_label"].value_counts(dropna=False))

print("\nLabel flag distributions:")

for col in label_cols[1:]:

    #print("\n" + col)

    print(master_table[col].value_counts(dropna=False))


Validation label distribution:
validation_label
unlabeled               41280
neutral                  8183
disease_suspected         153
pathogenic_confirmed       88
Name: count, dtype: int64

Label flag distributions:
is_disease_suspected_dataset3
0    49533
1      171
Name: count, dtype: int64
is_neutral_dataset8
0    41420
1     8284
Name: count, dtype: int64
is_pathogenic_dataset9
0    49616
1       88
Name: count, dtype: int64


In [62]:
# ============================================================
# 4. Define new analysis groups
# ============================================================
# В этом подходе:
# - neutral_from_article используется как reference neutral set
# - article_pathogenic НЕ используется как training positive
# - disease_suspected НЕ используется как training positive
# - unlabeled/candidates будут только score set

master_table["analysis_group"] = "unlabeled_or_other"

master_table.loc[master_table["is_neutral_dataset8"] == 1, "analysis_group"] = "neutral_reference"

master_table.loc[
    (master_table["is_disease_suspected_dataset3"] == 1)
    & (master_table["is_neutral_dataset8"] != 1),
    "analysis_group"
] = "disease_suspected_posthoc"

master_table.loc[
    (master_table["is_pathogenic_dataset9"] == 1)
    & (master_table["is_neutral_dataset8"] != 1),
    "analysis_group"
] = "article_pathogenic_posthoc"

print("\nNew analysis group distribution:")
print(master_table["analysis_group"].value_counts(dropna=False))

# Проверяем пересечения
overlap_neutral_article_pathogenic = (
    (master_table["is_neutral_dataset8"] == 1)
    & (master_table["is_pathogenic_dataset9"] == 1)
)

overlap_neutral_disease_suspected = (
    (master_table["is_neutral_dataset8"] == 1)
    & (master_table["is_disease_suspected_dataset3"] == 1)
)

print("\nOverlap: neutral & article pathogenic:")
print(overlap_neutral_article_pathogenic.sum())

print("\nOverlap: neutral & disease suspected:")
print(overlap_neutral_disease_suspected.sum())


New analysis group distribution:
analysis_group
unlabeled_or_other            41280
neutral_reference              8284
article_pathogenic_posthoc       84
disease_suspected_posthoc        56
Name: count, dtype: int64

Overlap: neutral & article pathogenic:
4

Overlap: neutral & disease suspected:
99


In [63]:
audit_cols = id_cols + label_cols + ["analysis_group"]

master_table[audit_cols].to_csv(LABEL_AUDIT, sep="\t", index=False)

print("\nSaved label audit:")

print(LABEL_AUDIT)


Saved label audit:
../data/processed/neutral_scoring_prepared/neutral_scoring_label_audit_v1.tsv


In [64]:
# ============================================================
# 6. Define primary feature sets
# ============================================================

mlc_only_features = [
    "mlc_score",
]

population_features = [
    "gnomad_observed",
    "gnomad_homoplasmic_af",
    "gnomad_heteroplasmic_af",
    "gnomad_combined_af_simple",
    "gnomad_max_heteroplasmy",

    "helix_observed",
    "helix_af_hom",
    "helix_af_het",
    "helix_mean_arf",
    "helix_max_arf",
]

combined_features = mlc_only_features + population_features

all_primary_features = list(dict.fromkeys(combined_features))

missing_feature_cols = [
    col for col in all_primary_features
    if col not in master_table.columns
]

if missing_feature_cols:
    raise ValueError(f"Missing feature columns: {missing_feature_cols}")

print("\nPrimary features:")
for col in all_primary_features:
    print("-", col)


Primary features:
- mlc_score
- gnomad_observed
- gnomad_homoplasmic_af
- gnomad_heteroplasmic_af
- gnomad_combined_af_simple
- gnomad_max_heteroplasmy
- helix_observed
- helix_af_hom
- helix_af_het
- helix_mean_arf
- helix_max_arf


In [65]:
# ============================================================
# 7. Numeric conversion
# ============================================================

work = master_table.copy()

for col in all_primary_features:
    work[col] = pd.to_numeric(work[col], errors="coerce")

# ------------------------------------------------------------
# Population features:
# missing means not observed / no recorded AF in the database
# ------------------------------------------------------------

zero_fill_population_cols = [
    "gnomad_observed",
    "gnomad_homoplasmic_af",
    "gnomad_heteroplasmic_af",
    "gnomad_combined_af_simple",
    "gnomad_max_heteroplasmy",

    "helix_observed",
    "helix_af_hom",
    "helix_af_het",
    "helix_mean_arf",
    "helix_max_arf",
]

work[zero_fill_population_cols] = work[zero_fill_population_cols].fillna(0.0)

work["gnomad_observed"] = work["gnomad_observed"].astype(int)
work["helix_observed"] = work["helix_observed"].astype(int)

# ------------------------------------------------------------
# Do not impute mlc_score arbitrarily
# ------------------------------------------------------------

before = work.shape[0]
work = work[work["mlc_score"].notna()].copy()
after = work.shape[0]

print(f"\nRows before mlc_score filtering: {before}")
print(f"Rows after mlc_score filtering:  {after}")
print(f"Removed rows:                   {before - after}")


Rows before mlc_score filtering: 49704
Rows after mlc_score filtering:  49704
Removed rows:                   0


In [66]:
# ============================================================
# 8. Derived population features
# ============================================================

work["observed_in_any_population_db"] = (
    (work["gnomad_observed"] == 1)
    | (work["helix_observed"] == 1)
).astype(int)

work["observed_in_both_population_dbs"] = (
    (work["gnomad_observed"] == 1)
    & (work["helix_observed"] == 1)
).astype(int)

work["pop_af_hom_max"] = work[
    ["gnomad_homoplasmic_af", "helix_af_hom"]
].max(axis=1)

work["pop_af_het_max"] = work[
    ["gnomad_heteroplasmic_af", "helix_af_het"]
].max(axis=1)

work["pop_af_max"] = work[
    [
        "gnomad_homoplasmic_af",
        "gnomad_heteroplasmic_af",
        "helix_af_hom",
        "helix_af_het",
    ]
].max(axis=1)

work["pop_af_hom_sum"] = (
    work["gnomad_homoplasmic_af"]
    + work["helix_af_hom"]
)

work["pop_af_het_sum"] = (
    work["gnomad_heteroplasmic_af"]
    + work["helix_af_het"]
)

work["pop_af_sum"] = (
    work["pop_af_hom_sum"]
    + work["pop_af_het_sum"]
)

work["pop_max_heteroplasmy_or_arf"] = work[
    [
        "gnomad_max_heteroplasmy",
        "helix_mean_arf",
        "helix_max_arf",
    ]
].max(axis=1)

In [67]:
# ============================================================
# 9. Derived population features
# ============================================================

# ============================================================
# 9. Log-transformed population features
# ============================================================

eps = 1e-12

log_transform_cols = [
    "gnomad_homoplasmic_af",
    "gnomad_heteroplasmic_af",
    "gnomad_combined_af_simple",
    "helix_af_hom",
    "helix_af_het",
    "pop_af_hom_max",
    "pop_af_het_max",
    "pop_af_max",
    "pop_af_hom_sum",
    "pop_af_het_sum",
    "pop_af_sum",
]

for col in log_transform_cols:
    work[f"log10_{col}"] = np.log10(work[col] + eps)

In [68]:
# ============================================================
# 10. Final feature sets
# ============================================================

feature_set_mlc_only = [
    "mlc_score",
]

feature_set_population_only = [
    "gnomad_observed",
    "helix_observed",
    "observed_in_any_population_db",
    "observed_in_both_population_dbs",

    "gnomad_homoplasmic_af",
    "gnomad_heteroplasmic_af",
    "gnomad_combined_af_simple",
    "helix_af_hom",
    "helix_af_het",

    "pop_af_hom_max",
    "pop_af_het_max",
    "pop_af_max",
    "pop_af_hom_sum",
    "pop_af_het_sum",
    "pop_af_sum",

    "log10_gnomad_homoplasmic_af",
    "log10_gnomad_heteroplasmic_af",
    "log10_gnomad_combined_af_simple",
    "log10_helix_af_hom",
    "log10_helix_af_het",
    "log10_pop_af_hom_max",
    "log10_pop_af_het_max",
    "log10_pop_af_max",
    "log10_pop_af_hom_sum",
    "log10_pop_af_het_sum",
    "log10_pop_af_sum",

    "gnomad_max_heteroplasmy",
    "helix_mean_arf",
    "helix_max_arf",
    "pop_max_heteroplasmy_or_arf",
]

feature_set_combined = feature_set_mlc_only + feature_set_population_only

feature_sets = {
    "mlc_only": feature_set_mlc_only,
    "population_only": feature_set_population_only,
    "combined_mlc_population": feature_set_combined,
}

# Remove duplicates while preserving order
for name, cols in feature_sets.items():
    feature_sets[name] = list(dict.fromkeys(cols))

# Safety check
for name, cols in feature_sets.items():
    missing = [col for col in cols if col not in work.columns]
    if missing:
        raise ValueError(f"Feature set {name} has missing columns: {missing}")

    non_numeric = [
        col for col in cols
        if not pd.api.types.is_numeric_dtype(work[col])
    ]

    if non_numeric:
        raise ValueError(f"Feature set {name} has non-numeric columns: {non_numeric}")

    missing_values = work[cols].isna().sum()
    missing_values = missing_values[missing_values > 0]

    if len(missing_values) > 0:
        raise ValueError(
            f"Feature set {name} has missing values:\n{missing_values}"
        )

print("\nFeature sets:")
for name, cols in feature_sets.items():
    print(f"{name}: {len(cols)} features")


Feature sets:
mlc_only: 1 features
population_only: 30 features
combined_mlc_population: 31 features


In [69]:
# ============================================================
# 11. Save prepared scoring input
# ============================================================

all_feature_cols = sorted(
    set(
        feature_set_mlc_only
        + feature_set_population_only
        + feature_set_combined
    )
)

output_cols = (
    id_cols
    + label_cols
    + ["analysis_group"]
    + all_feature_cols
)

scoring_input = work[output_cols].copy()

print("\nScoring input shape:", scoring_input.shape)
print("\nAnalysis group distribution:")
print(scoring_input["analysis_group"].value_counts())

scoring_input.to_csv(OUTPUT_TABLE, sep="\t", index=False)

print("\nSaved scoring input:")
print(OUTPUT_TABLE)


Scoring input shape: (49704, 40)

Analysis group distribution:
analysis_group
unlabeled_or_other            41280
neutral_reference              8284
article_pathogenic_posthoc       84
disease_suspected_posthoc        56
Name: count, dtype: int64

Saved scoring input:
../data/processed/neutral_scoring_prepared/neutral_scoring_input_v1.tsv


In [70]:
# ============================================================
# 12. Save feature sets report
# ============================================================

with open(FEATURE_SETS_REPORT, "w") as f:
    f.write("Neutral scoring feature sets v1\n")
    f.write("===============================\n\n")

    f.write("Excluded from primary model:\n")
    f.write("- validation_label\n")
    f.write("- is_pathogenic_dataset9\n")
    f.write("- is_disease_suspected_dataset3\n")
    f.write("- variant_id\n")
    f.write("- position\n")
    f.write("- reference\n")
    f.write("- alternate\n")
    f.write("- substitution_type\n")
    f.write("- gene\n")
    f.write("- consequence\n")
    f.write("- exact coordinate features\n\n")

    for name, cols in feature_sets.items():
        f.write(f"{name}\n")
        f.write("-" * len(name) + "\n")
        for col in cols:
            f.write(f"{col}\n")
        f.write("\n")

print("\nSaved feature sets report:")
print(FEATURE_SETS_REPORT)


Saved feature sets report:
../data/processed/neutral_scoring_prepared/neutral_scoring_feature_sets_v1.txt


In [57]:
print(work.shape)

print(work["analysis_group"].value_counts())

print(work.head())

(49704, 85)
analysis_group
unlabeled_or_other            41280
neutral_reference              8284
article_pathogenic_posthoc       84
disease_suspected_posthoc        56
Name: count, dtype: int64
   position reference alternate         consequence  mlc_score variant_id  mlc_position_score  gnomad_observed gnomad_filter  AN  gnomad_homoplasmic_ac  gnomad_heteroplasmic_ac  \
0         1         G         T  intergenic_variant    0.65755     m.1G>T             0.65755                0           NaN NaN                    NaN                      NaN   
1         1         G         A  intergenic_variant    0.65755     m.1G>A             0.65755                0           NaN NaN                    NaN                      NaN   
2         1         G         C  intergenic_variant    0.65755     m.1G>C             0.65755                0           NaN NaN                    NaN                      NaN   
3         2         A         T  intergenic_variant    0.64832     m.2A>T          